<a href="https://colab.research.google.com/github/franz-hufana/Thesis-MobileNet/blob/main/Package-Docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Thesis System: MobileNetV2 and MobileNetV3 Wood Surface Defect Detection***

Import Necessary Tools

In [ ]:
import os
import gc
import cv2
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from imageio.v2 import imread
from skimage.transform import resize
from skimage.morphology import skeletonize, binary_closing, remove_small_objects, disk
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Large
from tensorflow.keras import backend as K
from matplotlib.patches import Rectangle
from google.colab import files

print("Import successfully!")

Import successfully!


**Choose among the two options**
1. Import via Google Drive
2. Import via Local Drive

In [ ]:
# Option 1: Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Palitan yung zip_path if iba yung zipfile name
zip_path = "/content/drive/MyDrive/Datasets.zip"

# Kung gusto mo palitan pwede rin
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted from Drive: {zip_path} to {extract_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted from Drive: /content/drive/MyDrive/Datasets.zip to /content/dataset


In [ ]:
# Option 2: Local Drive or Desktop/Laptop Drive

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
  zip_ref.extractall(extract_path)

print(f"Dataset extracted from upload: {zip_name} to {extract_path}")

**Labels the Datasets:**
* 4 classes in total

In [ ]:
folder_path = "/content/dataset/woods"

label_map = {
    'c' : 0,
    'f' : 1,
    'k' : 2,
    'n' : 3
}

class_names = ['cracks', 'fuzz', 'knothole', 'no defect']
num_classes = len(class_names)
img_size = (224, 224)

print(f"Class Names: {class_names}\nNumber of Classes: {num_classes}\nDataset Folder: {folder_path}")

Class Names: ['cracks', 'fuzz', 'knothole', 'no defect']
Number of Classes: 4
Dataset Folder: /content/dataset/woods


***Data Preprocessing:***
1. Checking and cleaning if its png, jpeg, jpg files and if its not it will be removed
2. Resizing
3. Converting image Grayscale and RGBA to RGB
4. Normalize pixel values to [-1, 1]


In [ ]:
def load_and_preprocess_dataset(folder_path, label_map, img_size=(224, 224)):

    image_paths = []

    for root, dirs, files_in_dir in os.walk(folder_path):
        for file_name in files_in_dir:
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(root, file_name))

    image_paths = sorted(image_paths)
    print(f"Total image files found: {len(image_paths)}")

    data = []
    labels = []
    file_names = []

    for file_path in image_paths:
        file_name = os.path.basename(file_path)
        prefix = file_name[0].lower()

        if prefix not in label_map:
            print(f"Skipping unknown file: {file_name}")
            continue

        try:
            img = imread(file_path)

            # Convert grayscale to RGB
            if img.ndim == 2:
                img = np.stack([img] * 3, axis=-1)

            # Convert RGBA to RGB
            elif img.ndim == 3 and img.shape[-1] == 4:
                img = img[:, :, :3]

            img = resize(
                img,
                img_size,
                anti_aliasing=True,
                preserve_range=True
            ).astype(np.float32)

            # MobileNet normalization to [-1, 1]
            img = (img / 127.5) - 1.0

            data.append(img)
            labels.append(label_map[prefix])
            file_names.append(file_name)

        except Exception as e:
            print(f"Error reading {file_name}: {e}")

    data = np.array(data, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    print("\nData Shape:", data.shape)
    print("Labels Shape:", labels.shape)

    print("\nSample Loaded Files:")
    for i in range(min(10, len(file_names))):
        print(f"{file_names[i]} ---> {class_names[labels[i]]}")

    return data, labels, file_names


data, labels, file_names = load_and_preprocess_dataset(
    folder_path=folder_path,
    label_map=label_map,
    img_size=img_size
)

Total image files found: 4000

Data Shape: (4000, 224, 224, 3)
Labels Shape: (4000,)

Sample Loaded Files:
c(1).png ---> cracks
c(10).png ---> cracks
c(100).png ---> cracks
c(101).png ---> cracks
c(102).png ---> cracks
c(103).png ---> cracks
c(104).png ---> cracks
c(105).png ---> cracks
c(106).png ---> cracks
c(107).png ---> cracks


***Splitting the Dataset for Training and Validating:***

* Can change the parameter if not satisfied to the split and results of training

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    data,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Training Data Shape :", x_train.shape)
print("Training Labels Shape:", y_train.shape)
print("Validation Data Shape:", x_val.shape)
print("Validation Labels Shape:", y_val.shape)

***Class Weights:***

* For Class or Dataset Imbalance



In [ ]:
unique_classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
class_weight_dict = dict(zip(unique_classes, class_weights_array))

print("Class Weights:")
for k, v in class_weight_dict.items():
    print(f"{class_names[k]}: {v:.4f}")

***Creating Data Augmentation Layer:***
* Add more Layers if applicable

In [ ]:
def get_data_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ], name="data_augmentation")

***Model Section:***
1. Preloading the Pretrained Models
2. Creating New Classifier head for transfer learning stage
3. Compile the New Classifier head

In [ ]:
def build_model(backbone_name, input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape, name="input_image")
    augmentation = get_data_augmentation()

    if backbone_name == "MobileNetV2":
        base_model = MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet"
        )

    elif backbone_name == "MobileNetV3":
        base_model = MobileNetV3Large(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet",
            include_preprocessing=False
        )

    else:
        raise ValueError("Unsupported model name")

    # Freeze base model for transfer learning stage
    base_model.trainable = False

    x = augmentation(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    outputs = Dense(num_classes, activation="softmax", name="wood_output")(x)

    model = Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_wood_model")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    return model

***Optional Step:***
* Fine-Tuning Function by unfreezing the last layers after initial training

In [ ]:
def fine_tune_model(model, backbone_name, unfreeze_last_layers=30, fine_tune_lr=1e-5):
    if backbone_name == "MobileNetV2":
        base_model_name = "mobilenetv2_1.00_224"
    elif backbone_name == "MobileNetV3":
        base_model_name = "MobilenetV3large"
    else:
        raise ValueError("Unsupported model")

    # Find the base model dynamically
    base_model = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer
            break

    if base_model is None:
        print("Base model not found. Fine-tuning skipped.")
        return model

    base_model.trainable = True

    for layer in base_model.layers[:-unfreeze_last_layers]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    print(f"\nFine-tuning enabled for {backbone_name}")
    print(f"Last {unfreeze_last_layers} layers are trainable.")
    return model

Train and Evaluate Models

In [ ]:
def train_and_evaluate(
    backbone_name,
    x_train, y_train,
    x_val, y_val,
    class_names,
    class_weight_dict,
    epochs=20,
    batch_size=32,
    do_fine_tune=False,

    # Optional or removed (#) to use
    # fine_tune_epochs=10
):
    print("\n" + "=" * 70)
    print(f"TRAINING {backbone_name}")
    print("=" * 70)

    K.clear_session()
    gc.collect()

    model = build_model(
        backbone_name=backbone_name,
        input_shape=(224, 224, 3),
        num_classes=len(class_names)
    )

    print(f"\nMODEL SUMMARY: {backbone_name}")
    model.summary()


# Frozen Base Model
    history_stage1 = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        verbose=2,
        class_weight=class_weight_dict
    )

    combined_history = {
        "accuracy": list(history_stage1.history["accuracy"]),
        "val_accuracy": list(history_stage1.history["val_accuracy"]),
        "loss": list(history_stage1.history["loss"]),
        "val_loss": list(history_stage1.history["val_loss"])
    }





In [ ]:
# Use if Fine-tuning is available
if do_fine_tune:
        model = fine_tune_model(
            model=model,
            backbone_name=backbone_name,
            unfreeze_last_layers=30,
            fine_tune_lr=1e-5
        )

        history_stage2 = model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=fine_tune_epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=2,
            class_weight=class_weight_dict
        )

        combined_history["accuracy"] += list(history_stage2.history["accuracy"])
        combined_history["val_accuracy"] += list(history_stage2.history["val_accuracy"])
        combined_history["loss"] += list(history_stage2.history["loss"])
        combined_history["val_loss"] += list(history_stage2.history["val_loss"])

In [ ]:
# Validation Prediction

    val_probs = model.predict(x_val, batch_size=batch_size, verbose=0)
    val_preds = np.argmax(val_probs, axis=1)

    cm = confusion_matrix(y_val, val_preds)

    print(f"\nCLASSIFICATION REPORT: {backbone_name}")
    print(classification_report(y_val, val_preds, target_names=class_names))

    macro_f1 = f1_score(y_val, val_preds, average='macro')
    micro_f1 = f1_score(y_val, val_preds, average='micro')
    weighted_f1 = f1_score(y_val, val_preds, average='weighted')

    metrics_dict = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "val_accuracy_last": combined_history["val_accuracy"][-1],
        "val_loss_last": combined_history["val_loss"][-1]
    }

    return model, combined_history, cm, metrics_dict